# LightGBM (CPU) - fe_v3 minus the two high-cardinality crosses

Kaggle port of `Experiments.ipynb` keeping **five** engineered features -
`TotalDiscount`, `PriceRatio`, `AverageMonthly`, `contract_x_payment`, `contract_x_internet` -
and dropping only the two highest-cardinality crosses, `service_profile` (~330 levels) and
`tenure_bin_x_contract`. LightGBM is tuned by a 60-trial Optuna study and refit with 5-fold
CV through the shared experiment harness.

**Why CPU (not GPU):** LightGBM's OpenCL GPU backend only offloads histogram construction,
which is negligible on this ~25-feature dataset, so a T4 sits near-idle and is often *slower*
than CPU. The study instead runs on all the instance's vCPUs - one single-threaded trial per
core - which is the fastest real option for LightGBM on data this shape.

**Settings (right sidebar):** Accelerator -> **None / CPU** is sufficient; Internet -> On
(for `git clone` + `pip`); Add Input -> playground-series-s6e3. A GPU instance also runs but
wastes quota.

> WARNING: Not verified against the live Kaggle environment. Run a ~4-trial study first to
> check per-trial timing before the full Save & Run All.

In [ ]:
# List attached inputs (confirm the competition data is mounted).
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import os, sys, subprocess

REPO_URL  = "https://github.com/biswajit-nag/Predict-Customer-Churn.git"
REPO_ROOT = "/kaggle/working/Predict-Customer-Churn"

if not os.path.exists(REPO_ROOT):
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)

os.chdir(REPO_ROOT)                       # CWD = repo root (fixes data paths + git_info)
# Put the clone FIRST on sys.path so its `src` wins over any other module named `src`,
# and drop a possibly-stale `src` cached by an earlier cell. (A plain
# `if REPO_ROOT not in sys.path` guard can leave a shadowing `src` ahead of ours.)
sys.path.insert(0, REPO_ROOT)
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]
print("CWD:", os.getcwd())

In [ ]:
# Kaggle's image already ships a CPU LightGBM; only optuna may be missing.
!pip install -q optuna

import os
import lightgbm
print("lightgbm:", lightgbm.__version__)
print("CPU cores:", os.cpu_count())

In [ ]:
# Quick sanity fit to confirm LightGBM imports and trains before the full study.
import numpy as np
from lightgbm import LGBMClassifier

_Xs = np.random.rand(2000, 6)
_ys = (np.random.rand(2000) > 0.5).astype(int)
LGBMClassifier(n_estimators=10, verbose=-1).fit(_Xs, _ys)
print("LightGBM CPU OK")

In [ ]:
# data/processed/*.parquet are git-ignored, so absent from the clone. Rebuild the
# NATIVE (category-dtype) frames from the attached competition CSVs - LightGBM splits
# on the category columns directly, so this is the encoding Experiments.ipynb uses.
import shutil
from pathlib import Path

raw_dir = Path(REPO_ROOT) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
for f in ("train.csv", "test.csv"):
    shutil.copy(f"/kaggle/input/competitions/playground-series-s6e3/{f}", raw_dir / f)

from src.data import prepare_data
train_df, test_df = prepare_data(encoding='native', force=True)
print(f'Loaded native: train_df {train_df.shape}, test_df {test_df.shape}')

In [ ]:
import json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from src.tracking import DATA_DIR, RUNS_DIR, RUNS_CSV
from src.cv import run_cv_experiment, save_experiment

### Feature engineering

keeps TotalDiscount, PriceRatio, AverageMonthly, contract_x_payment, contract_x_internet (5 features; drops service_profile + tenure_bin_x_contract)

In [ ]:
# Row-wise (stateless) feature engineering - keeps TotalDiscount, PriceRatio, AverageMonthly, contract_x_payment, contract_x_internet (5 features; drops service_profile + tenure_bin_x_contract)
# Only leakage-safe row-wise transforms here; the category crosses are left as raw
# strings and split on natively per-fold by LightGBM (no fitted encoder up front).
DATA_VERSION = 'fe_nocross5_native'  # distinct parquet cache for this feature subset


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['TotalDiscount']  = df['TotalCharges'] - df['MonthlyCharges'] * df['tenure']
    df['PriceRatio']     = df['MonthlyCharges'] * df['tenure'] / df['TotalCharges']
    df['AverageMonthly'] = df['TotalCharges'] / df['tenure']
    df['contract_x_payment'] = (df['Contract'].astype(str) + ' | '
                                + df['PaymentMethod'].astype(str)).astype('category')
    df['contract_x_internet'] = (df['Contract'].astype(str) + ' | '
                                 + df['InternetService'].astype(str)).astype('category')
    return df


# Cache engineered parquets per DATA_VERSION (rebuilt on-platform here).
fe_train_path = DATA_DIR / f'train_df_{DATA_VERSION}.parquet'
fe_test_path  = DATA_DIR / f'test_df_{DATA_VERSION}.parquet'

if fe_train_path.exists() and fe_test_path.exists():
    train_df = pd.read_parquet(fe_train_path)
    test_df  = pd.read_parquet(fe_test_path)
    print(f'Loaded cached FE: {DATA_VERSION}')
else:
    train_df = engineer_features(train_df)
    test_df  = engineer_features(test_df)
    pq.write_table(pa.Table.from_pandas(train_df, preserve_index=False), fe_train_path)
    pq.write_table(pa.Table.from_pandas(test_df,  preserve_index=False), fe_test_path)
    print(f'Computed and cached FE: {DATA_VERSION}')

In [ ]:
# Refresh feature list and design matrices from the engineered dataframes.
encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X_train = train_df[encoded_features]
y_train = train_df['Churn']
X_test  = test_df[encoded_features]
print(f'X_train: {X_train.shape}  X_test: {X_test.shape}  features: {len(encoded_features)}')

### Optuna study - LightGBM (CPU)

60 trials, 3-fold inner CV, ROC-AUC objective, parallelised across the instance's vCPUs
(one single-threaded LightGBM per core). Pin `run_config['params']` to `lgbm_study.best_params`,
so run this cell before the run cell.

In [ ]:
# --- Optuna LightGBM study on CPU ---
# LightGBM is the wrong tool to push onto a T4 here (its OpenCL GPU backend only offloads
# histogram construction - negligible on ~25 features), so the study runs on CPU. To use all
# vCPUs we run N_JOBS trials concurrently and cap each LightGBM to ONE thread (n_jobs=1 in the
# objective's params), so total threads == cores (no oversubscription). cross_val_score stays
# n_jobs=1 so a trial's 3 folds run sequentially within that one-thread budget. The final
# 5-fold refit (run-config cell) omits n_jobs, so it uses all cores.
#
# Search space matches Experiments.ipynb: coupled num_leaves<=2**max_depth, plus the
# four categorical regularizers (cat_smooth, cat_l2, min_data_per_group, max_cat_threshold)
# acting on the contract_x_* crosses.
import os
import optuna
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

N_JOBS     = os.cpu_count() or 4   # Kaggle CPU / T4x2 instances expose ~4 vCPUs
LGBM_FIXED = dict(subsample_freq=1, verbose=-1)


def lgbm_catreg_objective(trial):
    max_depth = trial.suggest_int('max_depth', 3, 12)
    params = {
        'n_estimators':       trial.suggest_int('n_estimators', 400, 1500),
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':          max_depth,
        'num_leaves':         trial.suggest_int('num_leaves', 4, min(2 ** max_depth, 255)),
        'min_child_samples':  trial.suggest_int('min_child_samples', 10, 200),
        'subsample':          trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_split_gain':     trial.suggest_float('min_split_gain', 0.0, 1.0),
        'cat_smooth':         trial.suggest_float('cat_smooth', 1.0, 500.0, log=True),
        'cat_l2':             trial.suggest_float('cat_l2', 0.1, 500.0, log=True),
        'min_data_per_group': trial.suggest_int('min_data_per_group', 10, 1000, log=True),
        'max_cat_threshold':  trial.suggest_int('max_cat_threshold', 8, 256, log=True),
        'n_jobs':             1,   # one thread/trial; the study runs N_JOBS trials at once
        **LGBM_FIXED,
    }
    scores = cross_val_score(LGBMClassifier(**params), X_train, y_train,
                             cv=inner_cv, scoring='roc_auc')
    return scores.mean()


lgbm_study = optuna.create_study(
    study_name='lgbm-catreg-cpu',
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)

# Warm-start from the known-good optimum (run 20260610-123628-24ca03), num_leaves 241->16
# (identical model at depth 4) and the cat regularizers at LightGBM defaults.
lgbm_study.enqueue_trial({
    'n_estimators':       1281,
    'learning_rate':      0.04023698297756047,
    'max_depth':          4,
    'num_leaves':         16,
    'min_child_samples':  97,
    'subsample':          0.6891222073964552,
    'colsample_bytree':   0.608154415222902,
    'reg_alpha':          8.812432213377056,
    'reg_lambda':         0.00011799392947671802,
    'min_split_gain':     0.23233379023930362,
    'cat_smooth':         10.0,
    'cat_l2':             10.0,
    'min_data_per_group': 100,
    'max_cat_threshold':  32,
})

# SMOKE-TEST FIRST: drop n_trials to ~4 to check per-trial time before the full 60-trial run.
lgbm_study.optimize(lgbm_catreg_objective, n_trials=60, n_jobs=N_JOBS, show_progress_bar=True)

print(f'Best inner-CV ROC AUC: {lgbm_study.best_value:.6f}  (trial {lgbm_study.best_trial.number})')
for k, v in lgbm_study.best_params.items():
    print(f'  {k:18s} {v}')

### Run configuration

In [ ]:
# --- Run config ---
# LightGBM on the engineered native features. best_params carries only the tuned
# hyperparameters (its n_jobs=1 was a study-only setting and is NOT in best_params, so the
# factory's LGBMClassifier defaults to all cores); the fixed flags (LGBM_FIXED) are merged
# back in by the model factory. The final 5-fold CV runs its folds sequentially (src/cv.py).
from lightgbm import LGBMClassifier

X_test = test_df[encoded_features]   # LightGBM remaps test categories by value - no realign needed

run_config = {
    'model_factory': lambda params: LGBMClassifier(**params, **LGBM_FIXED),
    'params':        lgbm_study.best_params,
    'metric':        accuracy_score,
    'metric_name':   'accuracy',
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           'lgbm-cpu-fe-nocross5',
    'notes':         'LightGBM GBDT (CPU) on Kaggle, FE = fe_v3 minus the two high-cardinality crosses (kept: TotalDiscount, PriceRatio, AverageMonthly, contract_x_payment, contract_x_internet; dropped: service_profile, tenure_bin_x_contract). Best params from a 60-trial Optuna study (3-fold inner CV, ROC-AUC) parallelised across vCPUs (one single-threaded trial per core); trial 0 seeded from run 20260610-123628-24ca03. Data regenerated on-platform; data_hash differs from local runs. Notebook: kaggle/predict-customer-churn-lgbm-cpu-nocross5.ipynb.',
    'parent_run_id': '20260610-123628-24ca03',
    'save_models':   False,
    'data_version':  DATA_VERSION,
}

In [ ]:
# Step 1 - Run the experiment (fits 5 folds, prints OOF accuracy + ROC-AUC).
result = run_cv_experiment(run_config, X_train, y_train, X_test, encoded_features)

In [ ]:
# Step 2 - Save the run (review the OOF ROC-AUC above first).
run_id = save_experiment(result)

### Build a submission (optional)

`test_proba_mean` is the fold-bagged churn probability for the full test set. The competition metric is ROC-AUC, so submit the probability directly.

In [ ]:
submission = pd.DataFrame({
    'id':    test_df['id'],
    'Churn': result['artifacts']['test_proba_mean'],
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission.head())
print('wrote /kaggle/working/submission.csv', submission.shape)

### Bundle run artifacts + source notebook into one zip

Zips the run directory, `runs.csv`, and the source `.ipynb` (committed in `kaggle/` of the cloned repo) into a single archive on the Output tab.

In [ ]:
import shutil
from pathlib import Path
from src.tracking import RUNS_DIR, RUNS_CSV

BUNDLE = Path('/kaggle/working/bundle')
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)

# 1) heavy run artifacts (params, oof_proba, test_proba_*, metrics, env, git diff)
shutil.copytree(RUNS_DIR / run_id, BUNDLE / 'runs' / run_id)
# 2) the master index row
shutil.copy(RUNS_CSV, BUNDLE / 'runs.csv')
# 3) source notebook committed in the cloned repo
src_nb = Path(REPO_ROOT) / 'kaggle' / 'predict-customer-churn-lgbm-cpu-nocross5.ipynb'
shutil.copy(src_nb, BUNDLE / src_nb.name)
print('bundled notebook:', src_nb.name)

archive = shutil.make_archive(f'/kaggle/working/{run_id}_bundle', 'zip', BUNDLE)
print('wrote', archive)